<a href="https://colab.research.google.com/github/etcex2969-spec/-AIFFEL_quest_eng/blob/main/Trading_Insight_Orchestrator_v4_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

def setup_environment():
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/your_project_folder')
    print("Google Drive 마운트 및 작업 디렉토리 설정 완료")

setup_environment()


In [ ]:
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')

# 마운트된 경로 확인 및 파일 읽기/쓰기 예시
file_path = '/content/drive/MyDrive/sample_data.txt'

# 파일 쓰기
with open(file_path, 'w') as f:
    f.write("This is a sample text file.")

# 파일 읽기
with open(file_path, 'r') as f:
    content = f.read()
print(content)


In [ ]:
!pip install -q yfinance
import yfinance as yf

# 엔비디아 데이터 ! (2025년부터 현재 2026년까지)
df = yf.download("NVDA", start="2025-01-01", end="2026-06-01")
df.to_csv("nvda_financial_data.csv") # 공으 공식 데이터셋 생성 완료!

In [ ]:
from langgraph.graph import StateGraph, END
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS
from langchain.llms import OpenAI
import re

class AgentState(dict):
    mcp_context: str = ""
    analyst_opinion: str = ""
    risk_assessment: str = ""
    final_report: str = ""
    user_feedback: str = ""
    feedback_category: str = ""
    model_version: str = "v3.0"

# 벡터 DB 및 LLM 초기화 (RAG 구성)
vectorstore = FAISS.load_local("faiss_index")
llm = OpenAI(model="gpt-4o-mini")
retrieval_qa = RetrievalQA(llm=llm, retriever=vectorstore.as_retriever())

# 1) MCP 데이터 수집
def research_agent(state: AgentState) -> AgentState:
    state.mcp_context = "MCP 데이터 수집 완료"
    return state

# 2) 분석가 의견 생성 (RAG 활용)
def analyst_agent(state: AgentState) -> AgentState:
    query = state.mcp_context
    answer = retrieval_qa.run(query)
    state.analyst_opinion = answer
    return state

# 3) 위험 평가
def risk_agent(state: AgentState) -> AgentState:
    state.risk_assessment = "위험 평가 완료"
    return state

# 4) 최종 보고서 작성
def report_agent(state: AgentState) -> AgentState:
    state.final_report = f"{state.analyst_opinion}\n{state.risk_assessment}"
    return state

# 5) 피드백 자동 분류
def feedback_agent(state: AgentState) -> AgentState:
    fb = state.user_feedback.lower()
    if re.search(r"(좋|만족|훌륭|감사)", fb):
        state.feedback_category = "positive"
    elif re.search(r"(나쁨|불만|오류|문제)", fb):
        state.feedback_category = "negative"
    else:
        state.feedback_category = "neutral"
    return state

# 워크플로우 구성
workflow = StateGraph(AgentState)
workflow.add_node("Research_Agent", research_agent)
workflow.add_node("Analyst_Agent", analyst_agent)
workflow.add_node("Risk_Agent", risk_agent)
workflow.add_node("Report_Agent", report_agent)
workflow.add_node("Feedback_Agent", feedback_agent)

workflow.set_entry_point("Research_Agent")
workflow.add_edge("Research_Agent", "Analyst_Agent")
workflow.add_edge("Analyst_Agent", "Risk_Agent")
workflow.add_edge("Risk_Agent", "Report_Agent")
workflow.add_edge("Report_Agent", "Feedback_Agent")
workflow.add_edge("Feedback_Agent", END)

app = workflow.compile()

# 실행 예시
state = AgentState()
state.user_feedback = "보고서가 매우 만족스럽습니다."
final_state = app.run(state)

print(final_state.final_report)
print(final_state.feedback_category)


 <아키텍처 개요>
LangChain: LLM 체인, 에이전트, 메모리, 벡터 DB 등 자연어 처리 핵심 기능 제공
LangGraph: 상태 기반 워크플로우 및 다중 에이전트 협업 관리
5단계 추론: 단계별 데이터 수집 → 분석 → 위험 평가 → 최종 보고서 → 피드백 분류
RAG: 외부 지식 검색(벡터 DB)과 생성 모델 결합으로 정확도 및 신뢰도 향상


            <통합 아키텍처 설계>[사용자 요청]
      ↓
[LangGraph 워크플로우]
  ├─ Step1: MCP 데이터 수집 (Research Agent)
  ├─ Step2: 분석가 의견 생성 (Analyst Agent, LangChain 체인 호출)
  ├─ Step3: 위험 평가 (Risk Agent)
  ├─ Step4: 최종 보고서 작성 (Report Agent)
  ├─ Step5: 피드백 자동 분류 (Feedback Agent)
      ↓
[벡터 DB 검색 (RAG) ↔ LangChain LLM 생성]
      ↓
[결과 반환 및 피드백 수집]
      ↓
[파인튜닝 및 모델 버전 관리]


각 단계는 LangGraph의 노드로 구현되고, LangChain 체인/에이전트가 내부 작업 수행
RAG는 벡터 DB에서 관련 문서 검색 후 LangChain 생성 모델에 입력으로 제공
피드백은 자동 분류 후 파인튜닝 데이터로 활용, 모델 버전 관리 및 롤백 지원

 <추가 고려사항>
파인튜닝 및 모델 버전 관리: 피드백 데이터를 별도 DB에 저장 후 주기적 파인튜닝, 새 모델 버전 배포 및 롤백 지원
보안: API 키 비밀관리, 데이터 암호화, 접근 권한 최소화
모니터링: 호출량, 오류, 파인튜닝 상태 실시간 대시보드 구축
